In [ ]:
# Installe d'abord dans une cellule : !pip install transformers datasets torch accelerate scikit-learn

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Choisis le modèle à tester :
# 1. "SI2M-Lab/DarijaBERT"
# 2. "UBC-NLP/MARBERTv2"
# 3. "aubmindlab/bert-base-arabertv02"
MODEL_NAME = "SI2M-Lab/DarijaBERT"

df = pd.read_csv("darija_clean.csv")
labels_list = sorted(df['label'].unique())
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for i, l in enumerate(labels_list)}
df['label_id'] = df['label'].map(label2id)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["clean_text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'})).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'})).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels_list), id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

args = TrainingArguments(
    output_dir="./results_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()
print("Évaluation finale :", trainer.evaluate())